# 2 · From words to genres — keywords, similarity, and clustering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexsosn/ugarit-dh-workshop/blob/master/notebooks/2_similarity_clustering.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alexsosn/ugarit-dh-workshop/master?labpath=notebooks%2F2_similarity_clustering.ipynb)


We turn every tablet into a vector of its vocabulary, project the vectors onto a 2-D map, and colour the points with editorial teaching labels. Then we test what the picture seems to show.

By the end, you should be able to distinguish four moves: **description** (keywords), **projection** (UMAP/PCA), **unsupervised clustering** (KMeans), and **supervised validation** (cross-validated classification). The map is an invitation to close reading—not a result by itself.

## Setup


In [ ]:
# === SETUP — run me first ===
import os, sys, subprocess, warnings
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")

if "google.colab" in sys.modules:                      # we're on Colab
    REPO_URL = "https://github.com/alexsosn/ugarit-dh-workshop.git"
    REPO_DIR = "/content/ugarit-dh-workshop"
    if not os.path.isdir(REPO_DIR):                    # clone the repo once
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))      # work from notebooks/
    # Colab already ships numpy/pandas/scikit-learn/matplotlib/plotly/networkx;
    # pyarrow for parquet support
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn", "pyarrow"], check=False)

# make workshop_tools importable (we run from the notebooks/ folder)
sys.path.insert(0, os.path.abspath(".."))

Let's load the texts and see what's inside each text object.

In [ ]:
from workshop_tools.loader import load_texts

texts = load_texts()     # Downloads/caches the CUC HuggingFace parquet if needed
print(f"Loaded {len(texts)} tablets — genre of the first one: {texts[0]['genre']}")
list(texts[0].keys())

## Where do the genre labels come from?

The **KTU catalogue** groups alphabetic texts by its first number:

- **1.** literary and religious texts
- **2.** letters
- **3.** legal texts
- **4.** economic or administrative texts
- **5.** scribal exercises
- **6.** inscriptions on seals, labels, ivories, etc.
- **7.** unclassified texts
- **8.** illegible tablets and uninscribed fragments
- **9.** unpublished texts

Our loader turns those catalogue groups into **coarse teaching labels** and adds finer labels for a conservative list of well-known texts (myth, epic, ritual, divination, god-list). These are editorial/heuristic labels, not neutral ground truth. We will ask whether vocabulary carries their signal—not whether an algorithm can reveal an uncontested ancient taxonomy.

In [ ]:
# Select only texts with at least 30 tokens for clustering
sample = [t for t in texts if len(t["tokens"]) >= 30]

# What's the genre of each of the selected texts?
genres = [t["genre"] for t in sample]

print(f"Selected {len(sample)} tablets with at least 30 tokens.")
print(f"Genres: {set(genres)}")

### Genre balance of the mapped tablets

Before the map: *how many* tablets of each genre are we actually projecting? The classes are **imbalanced** (letters dominate) — worth remembering when reading how tight or sparse each colour's cluster looks.

In [ ]:
import pandas as pd, plotly.express as px
from collections import Counter
gd = pd.Series(Counter(genres)).sort_values()
fig = px.bar(x=gd.values, y=gd.index, orientation="h", text=gd.values,
             color=gd.index, title="Genre balance of the mapped tablets")
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False,
                  xaxis_title="tablets", yaxis_title="",
                  margin=dict(l=120, r=30, t=50, b=30),
                  paper_bgcolor="white", plot_bgcolor="white")
fig.show()

In [ ]:
from workshop_tools.loader import corpus_as_documents
labels, docs = corpus_as_documents(sample)
print(labels[:10])  # print the first 10 labels
print(docs[:5])  # print the first 5 documents

## What are the most common words?

The most common words in any language are so-called "stop-words": prepositions, conjunctions, and other small lexemes.

In [ ]:
from workshop_tools.loader import token_counts
token_counts(sample).most_common(15)

## Turn each tablet into a TF-IDF vector

**Term frequency–inverse document frequency (TF-IDF)** gives a high weight to a form that is frequent in one tablet but uncommon across the other tablets. Very common particles are therefore down-weighted. Each tablet becomes a row, each distinct written form a column, and each cell a weight: a point in a high-dimensional *vocabulary space*.

For form $t$ in tablet $d$, scikit-learn's defaults compute

$$\operatorname{tfidf}(t,d)=\operatorname{count}(t,d)\left[\ln\left(\frac{1+N}{1+\operatorname{df}(t)}\right)+1\right],$$

where $N$ is the number of tablets and $\operatorname{df}(t)$ is the number of
tablets containing the form. The added 1s prevent division by zero. Each tablet
vector is then scaled to length 1: divide every weight by
$\sqrt{\sum_j x_j^2}$. So repeating a rare form raises its weight, appearing in
many tablets lowers it, and a long tablet does not win merely by having more words.

**Tiny example.** If a form occurs 3 times in one of $N=100$ tablets but appears
in only 4 tablets overall, its unnormalized weight is
$3[\ln(101/5)+1]\approx12.0$. A form in all 100 tablets receives only $3$.

This model counts **written forms**, not lemmas or meanings. Damage, restorations, spelling, tablet length, proper names, and the lack of morphological annotation all affect the result.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer(token_pattern=r"[^\s]+")
X = vec.fit_transform(docs).toarray()
print(f"{X.shape[0]} tablets in a {X.shape[1]}-word space")

### What the matrix looks like

The first ten rows and columns show the underlying numbers. The image below shows the whole matrix: tablets are rows and forms are columns. Most cells are zero because any one tablet uses only a small part of the corpus vocabulary; brighter cells carry more weight.

In [ ]:
X[:10, :10]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.imshow(np.log1p(X * 100), aspect='auto', cmap='magma')
plt.xlabel('word-form features')
plt.ylabel('tablets')
plt.title('Sparse TF-IDF matrix (log-scaled for visibility)')
plt.colorbar(label='log-scaled TF-IDF')
plt.tight_layout()

> **Do not read clusters from this heatmap.** Feature columns are in vocabulary order, not semantic order. Its job is simply to make sparsity visible before we compare tablets mathematically.

### Distribution of non-zero weights

A histogram shows that most non-zero weights are modest and a small tail is large. Those high-weight forms are candidates for a tablet's keywords—not automatic translations or interpretations.

In [ ]:
nonzero = X[X > 0]
print(f"{nonzero.size} non-zero TF-IDF values out of {X.size}")

plt.figure(figsize=(10, 4))
plt.hist(nonzero, bins=100, edgecolor="black")
plt.title("Histogram of non-zero TF-IDF values")
plt.xlabel("TF-IDF score")
plt.ylabel("Frequency")
plt.tight_layout()

## Top keywords per tablet

In [ ]:
import numpy as np
feat = np.array(vec.get_feature_names_out())

def top_kw(i, n=6):
    row = X[i].ravel()
    idx = row.argsort()[::-1][:n]
    return [feat[j] for j in idx if row[j] > 0]

rng = np.random.default_rng(42)
for i in rng.choice(len(sample), 10, replace=False):
    t = sample[i]
    print(f'{t["ktu"]:7s} [{t["genre"]:11s}] {top_kw(i)}')

## Top keywords per genre (UDB)

The optional UDB tables attach an editor-supplied genre to a much larger set of tablets. We concatenate the tablets in each genre and rank forms that are frequent there but less frequent in the other genres.

Read the output as **candidates for checking**. Personal/divine names, numerals, formulae, editorial artifacts, and common particles may all rank highly. A high score does not supply a lemma or translation, and the PDF parser may preserve tokens that are not part of the Ugaritic wording.

In [ ]:
# === Characteristic keywords per UDB genre ===
# Needs the local UDB parquet (built from your own PDF):
#     python -m workshop_tools.build_udb_parquet
from workshop_tools.udb_loader import udb_available, udb_tablet_corpus

if not udb_available():
    print("UDB parquet not built — skipping the UDB keywords.\n"
          "Build it from your own PDF:  python -m workshop_tools.build_udb_parquet")
else:
    udb = udb_tablet_corpus("00")
    udb = udb[udb["genre"].notna() & (udb["n_tokens"] > 0)].copy()
    udb["doc"] = udb["tokens"].map(" ".join)
    by_genre = udb.groupby("genre")["doc"].apply(" ".join)   # one document per genre
    vec_g = TfidfVectorizer(token_pattern=r"[^\s]+")
    Xg = vec_g.fit_transform(by_genre.values)
    feat_g = np.array(vec_g.get_feature_names_out())
    counts_g = udb["genre"].value_counts()
    print("Top TF-IDF keywords per UDB genre (each genre = one document):\n")
    for i, g in enumerate(by_genre.index):
        row = Xg[i].toarray().ravel()
        top = [feat_g[j] for j in row.argsort()[::-1][:10] if row[j] > 0]
        print(f"{g:16s} (n={counts_g[g]:4d})  {', '.join(top)}")

### ✍️ Philological audit

Choose one genre and three high-scoring forms. For each, open a concordance or edition and record: (1) the form as written, (2) a proposed lemma and meaning with a source, (3) whether it is a name, particle, number, damaged/restored form, or content word, and (4) whether its contexts support the genre claim.

A transliteration-to-cuneiform converter only changes scripts; it does **not** translate. If an LLM proposes meanings, verify them in DULAT/a grammar and in line context.

## Squash that space down to 2 dimensions

We compare two tablet vectors with **cosine similarity**:

$$\cos(x,y)=\frac{x\cdot y}{\lVert x\rVert\,\lVert y\rVert},
\qquad d_{\mathrm{cos}}=1-\cos(x,y).$$

It measures the angle between vocabulary profiles: 1 means the same direction,
0 means no shared weighted vocabulary. Because TF-IDF already scales each vector
to length 1, cosine similarity is simply their dot product.

**UMAP** first builds a weighted nearest-neighbour graph from those cosine
distances, then places points in 2-D so nearby relationships are preserved as well
as possible. It does not preserve every distance, and its axes have no philological
meaning. If UMAP is unavailable, we fall back to **PCA**, which instead finds the
two perpendicular directions containing the most variance. PCA is deterministic
linear compression; UMAP is a nonlinear neighbourhood map. They are useful checks
on one another, not interchangeable proofs of clusters.

In [ ]:
import numpy as np
warnings.filterwarnings("ignore", message="FNV hashing is not implemented in Numba.*")
warnings.filterwarnings("ignore", message="n_jobs value 1 overridden.*")
try:
    import umap
    xy = umap.UMAP(n_components=2, n_neighbors=12, min_dist=0.25,
                   metric="cosine", random_state=42).fit_transform(X)
    METHOD = "UMAP"
except Exception as e:
    from sklearn.decomposition import PCA
    xy = PCA(n_components=2, random_state=42).fit_transform(X)
    METHOD = f"PCA (UMAP unavailable: {type(e).__name__})"
print("projected with", METHOD)

In [ ]:
# Shared map helpers live in workshop_tools so the notebook stays readable.
from workshop_tools.similarity_helpers import cuc_frame, hover_config, spotlight, udb_frame


## Visualizing the map in 2-D

UMAP sees only the TF-IDF vectors; the coordinates do **not** use the labels. We add colour afterward so we can ask whether similarly labelled tablets occupy similar neighbourhoods. Distance is approximate and the axes have no philological meaning.

In [ ]:
# ✍️ Spotlight one tablet — change this and re-run THIS cell.
# Refer to a tablet by KTU number, excavation (RS) number, or part of its title:
#   "KTU 1.96"   "RS 3.341"   "1.4"   "Baal"   "letter"
HIGHLIGHT = "KTU 1.96"   # contested genre: filed as "myth", but maybe an evil-eye charm — see the note below

import plotly.express as px

df = cuc_frame(sample, x=xy[:, 0], y=xy[:, 1])
fig = px.scatter(df, x="x", y="y", color="genre",
                 title=f"CUC tablets in vocabulary space ({METHOD})",
                 **hover_config())
fig.update_traces(marker=dict(size=12, opacity=0.85, line=dict(width=0.5, color="white")))
fig.update_layout(height=640, legend_title_text="genre", xaxis_title="", yaxis_title="")
spotlight(fig, df, HIGHLIGHT)
fig.show()

### Two tablets of contested genres

The colours above pretend every tablet belongs to only one genre. The genre of many tablets has been hotly debated in fact. Let's look at two examples.

**KTU 1.96 — myth, or magic?**

For almost forty years this little tablet was Exhibit A for the **cannibal warrior-goddess Anat**: *"She ate his flesh without a knife, she drank his blood without a cup."* Whole theories of Ugaritic *theophagy* grew from its opening word, read as *ʿnt*, "Anat." Then Pitard re-photographed the tablet and the first word turned out to be *ʿnn*, **not** *ʿnt* — a single cuneiform **wedge** separates the two. With Anat gone, del Olmo Lete re-read the whole text as an **incantation against the evil eye** (*ʿn* = "eye"; *ʿnn hlkt* ≈ Akkadian *īnu muttalliktu*, "the roaming evil eye"). The CUC catalogue now files it as *"Incantation against Evil Eye."* So is it cosmic myth or folk medicine? **A wedge decides.**

> Note the irony: our own `FINE_GENRE` table hard-codes `1.96 → myth` (the older reading), so the map paints it with the myths. *(Lewis, "The Disappearance of the Goddess Anat in KTU 1.96," Biblical Archaeologist 59/2, 1996: 115–121.)*

**KTU 5.9 (= RS 16.265) — a letter that's really homework.**  &nbsp;*(not in the CUC mirror — meet it on the UDB map in §7, where it's `HIGHLIGHT = "RS 16.265"`)*

It opens like a warm private letter between friends and asks not for slaves but for **a cup of wine**. Virolleaud published it as a real letter (*"Lettre d'Eštl à Mnn"*). But it shares one tablet with **three abecedaries and word-lists** — it's a **scribal exercise**, a student's notebook page, so KTU³ files it under chapter 5 (scribal texts), not chapter 2 (letters). Yogev & Yona go one step further: the trainee shows off **poetic word-pairs and parallelism**, making it a *playful poetic letter* — a window into an Ugaritic classroom. Letter, abecedary, or poem? **All three.** *(Yogev & Yona, "A Poetic Letter: The Ugaritic Tablet RS 16.265," Studi Epigrafici e Linguistici 31–32, 2014: 51–58.)*

**Why this matters for the map.** A genre map draws crisp coloured regions; real tablets sometimes sit on the borders — or in the wrong colour entirely. The gap between the tidy label and the messy tablet is exactly where the interesting philology lives.


## Which tablets are most alike?
The nearest neighbours of one tablet, by shared vocabulary.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
S = pd.DataFrame(cosine_similarity(X), index=labels, columns=labels)
TARGET = "1.96"   # evil eye charm
print(f"Closest to KTU {TARGET}:")
print(S[TARGET].drop(TARGET).sort_values(ascending=False).head())

## Let the machine group them blind—and score the result

`KMeans` clusters with **no** genre labels. It repeatedly assigns every tablet to
the nearest centroid and moves each centroid to the mean of its assigned tablets,
minimizing the within-cluster sum of squares:

$$\sum_i \lVert x_i-\mu_{c_i}\rVert^2.$$

For two length-1 TF-IDF vectors, squared Euclidean distance is
$2(1-\cos(x,y))$, so the starting geometry is closely related to cosine
similarity. Ordinary KMeans does not re-normalize its centroids, however, so it is
not exactly the same algorithm as spherical (cosine) KMeans.

We score the result in two different ways. **Adjusted Rand index** counts pairs of
tablets placed together/apart in both partitions and corrects for chance:
$\mathrm{ARI}=(\mathrm{RI}-E[\mathrm{RI}])/(\max(\mathrm{RI})-E[\mathrm{RI}])$.
A value of 1 means identical partitions and around 0 means chance-level agreement.
**Silhouette** ignores the editorial labels and uses the same Euclidean distance.
For tablet $i$, let $a(i)$ be its mean distance to its own cluster and $b(i)$ the
lowest mean distance to another cluster:
$s(i)=[b(i)-a(i)]/\max[a(i),b(i)]$. Near 1 is well separated, near 0 is a boundary,
and below 0 suggests the tablet may fit another cluster better. We report the mean.

Then we ask a different question. A supervised logistic classifier *is* shown the
labels during training. For each genre it learns a linear score
$b_g+w_g\cdot x$, but is evaluated on held-out tablets in three folds. If its mean
accuracy beats the majority-label baseline, vocabulary has predictive genre signal
even if the corpus does not form clean natural clusters.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

km = KMeans(n_clusters=4, random_state=0, n_init=10).fit(X)
grid = pd.crosstab(pd.Series(km.labels_, name="cluster"), pd.Series(genres, name="genre"))
display(grid)

ari = adjusted_rand_score(genres, km.labels_)
sil = silhouette_score(X, km.labels_)
majority = max(Counter(genres).values()) / len(genres)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
clf = LogisticRegression(max_iter=2000, class_weight="balanced")
accuracy = cross_val_score(clf, X, genres, cv=cv, scoring="accuracy")
print(f"Blind KMeans: ARI={ari:.3f}; silhouette={sil:.3f}")
print(f"Held-out classifier: {accuracy.mean():.1%} mean accuracy "
      f"(folds {accuracy.round(3)}); majority baseline={majority:.1%}")

## Discussion

With the bundled sample and fixed seeds, KMeans agreement is **partial** (ARI ≈ 0.34) and the clusters are diffuse (silhouette ≈ 0.03). Yet the held-out classifier reaches about **83% accuracy**, versus a majority baseline of about **33%**. Vocabulary therefore carries substantial label signal without resolving into four clean, natural islands.

That distinction is the result: projection is for exploration, clustering tests self-organization, supervised classification tests predictability, and **close reading** decides whether an apparent agreement or disagreement is historically meaningful.

## ✍️ Your turn
Edit one value below and re-run. Nothing here can break the notebook — if it goes sideways, just re-run the setup cell.


In [ ]:
# Re-run sections 1–3 after changing either of these.
warnings.filterwarnings("ignore", message="FNV hashing is not implemented in Numba.*")
warnings.filterwarnings("ignore", message="n_jobs value 1 overridden.*")
NEIGHBORS = 8        # ← UMAP n_neighbors: smaller = more local clumps, larger = smoother

sample = [t for t in texts if len(t["tokens"]) >= 30]
genres = [t["genre"] for t in sample]
labels, docs = corpus_as_documents(sample)
X = TfidfVectorizer(token_pattern=r"[^\s]+").fit_transform(docs)
try:
    import umap
    xy = umap.UMAP(n_components=2, n_neighbors=NEIGHBORS, min_dist=0.25,
                   metric="cosine", random_state=42).fit_transform(X.toarray())
    METHOD = "UMAP"
except Exception as e:
    from sklearn.decomposition import PCA
    xy = PCA(n_components=2, random_state=42).fit_transform(X.toarray()); METHOD = "PCA"
print(f"{len(sample)} tablets — now re-run the map cell (section 3) to see them.")
# Hint: add "god-list" or "epic" and watch whether they carve out their own corner.

### Make it 3-D

Two dimensions force the map to flatten things that may really sit apart. **Add a third axis** and rotate it with the mouse — genres that overlap in 2-D sometimes peel apart in 3-D (and sometimes the extra axis just spreads the same blob). This reuses your `NEIGHBORS` from the cell above, so tweak those and re-run.

**Spin it and ask:**
- Do **letters** stay one tight ball while **myth / ritual / divination** stretch along their own directions?
- Does the 3rd axis *reveal* a real split, or merely *inflate* the 2-D picture? Compare with the map in §3.
- Trust structure that survives **2-D, 3-D, and several `NEIGHBORS` values** — if a cluster only shows up once, be suspicious.

In [ ]:
# === ✍️ Your turn — the 3-D map (drag to rotate) ===
# ✍️ Spotlight one tablet — change this and re-run THIS cell.
# Refer to a tablet by KTU number, excavation (RS) number, or part of its title:
#   "KTU 1.96"   "RS 3.341"   "1.4"   "Baal"   "letter"
HIGHLIGHT = "KTU 1.96"   # contested reading/genre discussed above
import plotly.express as px
warnings.filterwarnings("ignore", message="FNV hashing is not implemented in Numba.*")
warnings.filterwarnings("ignore", message="n_jobs value 1 overridden.*")

sample3 = [t for t in texts if len(t["tokens"]) >= 30]
labels3, docs3 = corpus_as_documents(sample3)
X3 = TfidfVectorizer(token_pattern=r"[^\s]+").fit_transform(docs3)

try:
    import umap
    xyz = umap.UMAP(n_components=3, n_neighbors=NEIGHBORS, min_dist=0.25,
                    metric="cosine", random_state=42).fit_transform(X3.toarray())
    METHOD3 = "UMAP-3D"
except Exception as e:
    from sklearn.decomposition import PCA
    xyz = PCA(n_components=3, random_state=42).fit_transform(X3.toarray()); METHOD3 = "PCA-3D"

df3 = cuc_frame(sample3, x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2])
fig = px.scatter_3d(df3, x="x", y="y", z="z", color="genre",
                    title=f"CUC tablets in 3-D vocabulary space ({METHOD3}) — drag to rotate",
                    **hover_config(z=True))
fig.update_traces(marker=dict(size=5, opacity=0.85, line=dict(width=0.5, color="white")))
fig.update_layout(height=720, legend_title_text="genre",
                  scene=dict(xaxis_title="", yaxis_title="", zaxis_title=""))
spotlight(fig, df3, HIGHLIGHT, coords=("x", "y", "z"))
fig.show()

### 🔭 Going further — more to extract from this map

The genre map is the doorway, not the whole house. Each idea below is a small add-on:

1. **Stress-test the score.** Repeat the ARI, silhouette, and held-out classifier at several minimum lengths and random seeds. Report the distribution, not only the best run.
2. **Why does a cluster cluster?** Print each KMeans cluster's **top TF-IDF forms** (highest mean weight).
   Expect letters bound by address formulas (*l, rgm, thm, yslm*) and myth by divine names — the *vocabulary*
   behind the geometry.
3. **Who's misfiled?** For every tablet, find its **nearest genre centroid**; the tablets whose nearest
   centroid ≠ their catalogued genre are the machine's "I'd relabel this" list — hybrids, mixed tablets, or
   genuine miscatalogues worth a second look.
4. **Bridges between genres.** Points roughly equidistant from two centroids are genre-**blends** (a ritual
   studded with myth, a letter quoting a formula). List the closest few and read them.
5. **Real, or an artefact?** Re-project with **PCA** and **t-SNE** as well as UMAP, at several `n_neighbors`.
   Structure that survives every method and setting is trustworthy; a split that appears only once is probably
   noise — the critical-thinking counterpart to the 3-D view above.
6. **Sub-genres within a genre.** Cluster *only* the myth tablets: do the Baal-cycle texts (1.1–1.6) peel away
   from Kirta / Aqhat? There is structure below the genre level too.

## The same map, but on the UDB genres

The map above colours CUC tablets by the **coarse KTU-chapter heuristic** (chapter 1 = literary, 2 = letter, 3 = legal). The **Ugaritic Data Bank** instead ships a genre *chosen by editors* for each tablet — `Administrative`, `Ritual`, `Correspondence`, `Myth`, `Legal`, `Epic`, … — over a corpus ~5× larger than CUC.

If you built the local UDB parquet (`workshop_tools/build_udb_parquet`, from your own PDF), the next cell projects the UDB tablets the same way — TF-IDF → UMAP, now in **3-D** (drag to rotate) — but colours by UDB's own genre. Watch whether the big `Administrative` mass (which the KTU heuristic can't even name) separates from the literary genres, and whether `Correspondence` forms one tight ball like CUC's letters. Hover shows each tablet's KTU id and the coarse `ktu_genre` for comparison.

> If you skipped the UDB build, the cell prints a one-line note and the rest of the notebook is unaffected.

In [ ]:
# === The same map on the UDB corpus — coloured by UDB's *own* genre ===
# Needs the local UDB parquet (built from your own PDF):
#     python -m workshop_tools.build_udb_parquet
from workshop_tools.udb_loader import udb_available, udb_tablet_corpus
import plotly.express as px

# ✍️ Spotlight one tablet — change this and re-run THIS cell.
# Refer to a tablet by KTU number, excavation (RS) number, or part of its title:
#   "RS 16.265"   "KTU 1.96"   "Baal"   "letter"
HIGHLIGHT = "RS 16.265"   # = KTU 5.9: a "letter" that is really a scribal exercise — see the §3 note

if not udb_available():
    print("UDB parquet not built — skipping the UDB map.\n"
          "Build it from your own PDF:  python -m workshop_tools.build_udb_parquet")
else:
    udb = udb_tablet_corpus("00")                       # one row per tablet, edition "00"
    udb = udb[(udb["genre"].notna()) & (udb["n_tokens"] >= 30)].reset_index(drop=True)
    docs_u = [" ".join(toks) for toks in udb["tokens"]]
    Xu = TfidfVectorizer(token_pattern=r"[^\s]+").fit_transform(docs_u)
    print(f"{Xu.shape[0]} UDB tablets in a {Xu.shape[1]}-word space; "
          f"{udb['genre'].nunique()} curated genres")

    try:
        import umap
        xyu = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.25,
                        metric="cosine", random_state=42).fit_transform(Xu.toarray())
        METHOD_U = "UMAP-3D"
    except Exception as e:
        from sklearn.decomposition import PCA
        xyu = PCA(n_components=3, random_state=42).fit_transform(Xu.toarray())
        METHOD_U = f"PCA-3D ({type(e).__name__})"

    df_u = udb_frame(udb, x=xyu[:, 0], y=xyu[:, 1], z=xyu[:, 2])
    fig = px.scatter_3d(df_u, x="x", y="y", z="z", color="genre",
                        title=f"{len(udb)} UDB tablets in 3-D vocabulary space "
                              f"({METHOD_U}) — drag to rotate",
                        **hover_config(z=True))
    fig.update_traces(marker=dict(size=5, opacity=0.8, line=dict(width=0.4, color="white")))
    fig.update_layout(height=720, legend_title_text="UDB genre",
                      scene=dict(xaxis_title="", yaxis_title="", zaxis_title=""))
    spotlight(fig, df_u, HIGHLIGHT, coords=("x", "y", "z"))
    fig.show()